In [ ]:
import numpy as np
from scipy.integrate import trapezoid

# Constants
NA = 6.02214076e23           # Avogadro's number (mol^-1)
PREFAC = 3.7313e7            # Prefactor (cm³ mol⁻¹ s⁻¹)
Z1 = 2                       # Alpha particle charge
m_alpha = 4.00260325415      # Alpha particle mass in amu

def reduced_mass_amu(m1_amu, m2_amu):
    """Compute reduced mass in atomic mass units (amu)."""
    return (m1_amu * m2_amu) / (m1_amu + m2_amu)

def reaction_rate_single_T9(T9, energies, cross_sections, mu_amu):
    """Compute <σv> for a given T9 (GK)."""
    maxwellian = np.exp(-11.605 * energies / T9)
    integrand = energies * cross_sections * maxwellian
    integral = trapezoid(integrand, energies)
    rate = PREFAC * mu_amu**(-0.5) * T9**(-1.5) * integral
    return rate

def compute_reaction_rate_over_T9(energies, cross_sections, mu_amu):
    """Compute <σv> across a temperature grid from 0.001 GK to 11 GK."""
    T9_values = np.linspace(0.001, 11, 50)
    rates = np.array([
        reaction_rate_single_T9(T9, energies, cross_sections, mu_amu)
        for T9 in T9_values
    ])
    return T9_values, rates

def main():
    print("Enter energies in MeV separated by spaces:")
    energies = np.array(list(map(float, input().strip().split())))

    print("Enter cross sections in milli-barn separated by spaces:")
    cross_sections = np.array(list(map(float, input().strip().split())))

    if len(energies) != len(cross_sections):
        print("Error: The number of energies and cross sections must be equal.")
        return

    print("Enter target atomic number (Z2):")
    Z2 = int(input())

    print("Enter target mass in atomic mass units (amu):")
    mass_target = float(input())

    mu_amu = reduced_mass_amu(m_alpha, mass_target)
    print(f"\nReduced mass μ = {mu_amu:.5f} amu\n")

    T9_vals, rate_vals = compute_reaction_rate_over_T9(energies, cross_sections, mu_amu)

    # ---- Part 1: ln(<σv>) ----
    print(f"{'T9 (GK)':>10} {'ln(<σv>) (cm³ mol⁻¹ s⁻¹)':>40}")
    print("-" * 52)
    for T9, rate in zip(T9_vals, rate_vals):
        if rate > 0:
            ln_rate = np.log(rate)
            print(f"{T9:10.3f} {ln_rate:40.6f}")
        else:
            print(f"{T9:10.3f} {'-inf':>40}")  # ln(0) is undefined

    print("\n" + "="*52 + "\n")

    # ---- Part 2: Actual <σv> ----
    print(f"{'T9 (GK)':>10} {'<σv> (cm³ mol⁻¹ s⁻¹)':>40}")
    print("-" * 52)
    for T9, rate in zip(T9_vals, rate_vals):
        print(f"{T9:10.3f} {rate:40.6e}")

if __name__ == "__main__":
    main()


Enter energies in MeV separated by spaces:
5.025 5.105 5.185 5.265 5.317 5.344 5.423 5.493 5.568 5.643 5.718 5.793 5.865 5.935 6.005 6.075 6.145 6.215 6.285 6.355 6.425 6.495 6.565 6.635 6.703 6.768 6.833 6.898 6.963 7.028 7.093 7.158 7.223 7.288 7.353 7.418 7.473 7.545 7.605 7.665 7.725 7.785 7.845 7.905 7.965 8.025 8.085 8.145 8.205 8.265 8.325 8.385 8.445 8.505 8.563 8.618 8.673 8.728 8.782 8.837 8.892 8.947 9.002 9.057 9.112 9.167 9.222 9.277 9.332 9.387 9.442 9.497 9.552 9.607 9.662 9.717 9.768 9.82 9.87 9.92 9.97
Enter cross sections in milli-barn separated by spaces:
0.04 0.057 0.077 0.113 0.141 0.152 0.169 0.267 0.324 0.426 0.541 0.677 0.875 1.051 1.394 1.592 1.969 2.444 2.939 3.245 3.999 4.805 5.652 6.467 7.526 8.567 9.94 11.199 12.81 13.93 16.091 18.104 20.75 23.644 24.828 28.141 32.15 35.06 35.5 41.325 43.984 48.095 52.749 55.01 59.18 64.83 68.17 74.38 81.07 86.86 89.97 96.95 104.41 111.83 115.88 121.94 126.93 132.5 138.41 147.23 152.92 160.33 165.58 175.12 178.79 189.63 196